In [1]:
import pandas as pd
import glob
import os
from tqdm.auto import tqdm
import geopandas as gpd
from shapely import wkt
from math import ceil
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

In [2]:
def quarter_slices(start, end):
    """
    Given two pandas Timestamps (start < end), return a list of (slice_start, slice_end)
    pairs, each of which lies wholly within one 15-minute bucket (minute marks 00,15,30,45).
    """
    out = []
    current = start

    while current < end:
        # Find “floor” of current to the nearest 15 minutes, then add 15 min
        floor_q = current.floor("15T")
        next_bound = floor_q + pd.Timedelta(minutes=15)

        # If current is exactly on a quarter, jump to the next one
        if current == floor_q:
            next_bound = current + pd.Timedelta(minutes=15)

        # End of this chunk is either the next boundary or the original end
        chunk_end = min(next_bound, end)

        out.append((current, chunk_end))
        current = chunk_end
    return out

In [3]:
folder_path = "datos/4_velocidades_waze/irregularidades"
feather_files = glob.glob(os.path.join(folder_path, "*.parquet"))

In [4]:
mvi = gpd.read_file("datos/2_Estructura_via/estructura_via_completa/MVI.geojson")
mvi = mvi.to_crs("EPSG:4326")

In [5]:
mvi.columns = mvi.columns.str.lower()
mvi["id_mvi"] = mvi["mviciv"].astype(str) + "_" + mvi["mvisvia"].str.replace(" ", "")
mvi['contador'] = mvi.groupby('id_mvi').cumcount() + 1
mvi['id_mvi'] = mvi['id_mvi'] + '.' + mvi['contador'].astype(str)

buffer_distance = 4
mvi_buffered = mvi.to_crs("EPSG:3116").copy()
mvi_buffered['mvi_length'] = mvi_buffered.geometry.length
mvi_buffered['geometry'] = mvi_buffered.geometry.buffer(buffer_distance, cap_style="square")
mvi_buffered_buffer = mvi_buffered.set_geometry('geometry')
mvi_buffered_buffer = mvi_buffered_buffer.to_crs("EPSG:4326")


In [6]:
file_output_name = "datos_procesados/irregularidades_procesadas/"
# Crear carpeta de salida si no existe
os.makedirs(file_output_name, exist_ok=True)

In [7]:
df = pd.read_parquet(f'{file_output_name}irregs_proc_2024-06-30_2024-07-09.parquet')

In [8]:
df

,id_mvi,timeframe,ACCIDENT_MAJOR_count,ACCIDENT_MAJOR_nThumbsUp,ACCIDENT_MINOR_count,ACCIDENT_MINOR_nThumbsUp,HAZARD_ON_ROAD_CAR_STOPPED_count,HAZARD_ON_ROAD_CAR_STOPPED_nThumbsUp,HAZARD_ON_ROAD_CONSTRUCTION_count,HAZARD_ON_ROAD_CONSTRUCTION_nThumbsUp,...,causeType_,causeType_0,causeType_ACCIDENT,causeType_HAZARD,causeType_ROAD_CLOSED,trend,speed,jamLevel,severity,regularSpeed
0,0_FT.1,2024-07-03 11:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,1.000000,5.340000,4.000000,5.0,8.990000
1,0_FT.1,2024-07-03 12:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3,1,0,0,0,-0.794872,4.156667,4.000000,5.0,8.515641
2,0_FT.1,2024-07-04 12:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0.000000,5.950000,4.000000,5.0,11.970000
3,0_FT.1,2024-07-04 13:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2,0,0,0,0,-0.555556,7.488889,4.000000,5.0,10.875556
4,0_FT.1,2024-07-06 09:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,1.000000,7.830000,4.000000,5.0,12.960000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273808,9004747_TF.1,2024-07-09 12:45:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3,0,0,0,0,0.271186,10.755932,3.423729,5.0,37.032881
273809,9004747_TF.1,2024-07-09 13:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3,0,0,0,0,-1.000000,18.325000,2.852941,5.0,37.907059
273810,9004747_TF.1,2024-07-09 17:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,1,0,0,1.000000,8.370000,4.000000,5.0,31.620000
273811,9004747_TF.1,2024-07-09 17:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2,0,1,0,0,-0.162393,11.913077,3.418803,5.0,28.538291


In [ ]:
n_chunks = 10
chunk_size = len(feather_files)// n_chunks+1
filtrar = False

In [ ]:
%%time
for i in tqdm(range(chunk_size)):
    files_group = feather_files[i*n_chunks:(i+1)*n_chunks]
    min_file = min(files_group)
    max_file = max(files_group)
    start_date = min_file.split('/')[3].split('.')[0].split('_')[2]
    end_date = max_file.split('/')[3].split('.')[0].split('_')[2]
    output_file_name = f'{file_output_name}irregs_proc_{start_date}_{end_date}.parquet'
    irreg_gdf= []
    for archivo in tqdm(feather_files[i*n_chunks:(i+1)*n_chunks]):
        df = pd.read_parquet(archivo)
        df['geometry'] = df['geom_wkt'].apply(
            lambda geom: wkb.loads(geom) if isinstance(geom, bytes)
            else wkt.loads(geom) if isinstance(geom, str)
            else geom
        )
        df_exploded = df[["id", "alerts"]].explode("alerts")
        df_exploded = df_exploded.dropna(subset=["alerts"])
        alerts_normalized = pd.json_normalize(df_exploded["alerts"])
        alerts_normalized = alerts_normalized.drop(columns=['id'], errors='ignore')
        df_exploded = df_exploded.reset_index(drop=True)
        alerts_normalized = alerts_normalized.reset_index(drop=True)
        alerts_details = pd.concat(
            [ df_exploded[["id"]],   # shape: (total_dicts, 1)
            alerts_normalized      # shape: (total_dicts, N_keys)
            ],
            axis=1
        )
        if 'nThumbsUp' not in alerts_details.columns:
            alerts_details['nThumbsUp'] = 0
        alerts_details['count'] =1
        alerts_details = alerts_details.groupby(['id','subtype']).aggregate({'count':'sum','nThumbsUp':'sum'}).reset_index().pivot_table(index = 'id', columns = 'subtype', values = ['count','nThumbsUp']).fillna(0)
        alerts_details.columns = [f'{col[1]}_{col[0]}' if col[1]!='' else f'NOT_DEFINED_{col[0]}' for col in alerts_details.columns]
        alerts_details =  alerts_details.reset_index()
        df['timestamp_detectionDateMillis'] = pd.to_datetime(df['detectionDateMillis'], unit='ms')
        df['timestamp_updateDateMillis'] = pd.to_datetime(df['updateDateMillis'], unit='ms')
        df['timestamp_detectionDateMillis_ts_cot'] = pd.to_datetime(df['detectionDateMillis_ts_cot'], unit='ms')
        df['timestamp_updateDateMillis_ts_cot'] = pd.to_datetime(df['updateDateMillis_ts_cot'], unit='ms')
        df['timestamp_startTime_ts_cot'] = pd.to_datetime(df['startTime_ts_cot'], unit='ms')
        df['timestamp_endTime_ts_cot'] = pd.to_datetime(df['endTime_ts_cot'], unit='ms')
        df = pd.merge(df, alerts_details, on =['id'], how='left')
        df = df.drop(['country','updateDate','city','detectionDateMillis','endNode','street','highway','length','updateDateMillis','detectionDate','causeAlert', 'startNode','detectionDateMillis_ts_cot',
        'updateDateMillis_ts_cot', 'geom_wkt', 'startTime_ts_cot','type','alerts',
        'endTime_ts_cot', 'fecha_lectura', 'hora_lectura_dato'],axis=1)
        # break
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=4326)
        irreg_gdf.append(gdf)
    irreg_gdf = gpd.GeoDataFrame(pd.concat(irreg_gdf, ignore_index=True).fillna(0), crs=4326)
    irreg_gdf["irreg_id"] = irreg_gdf.index
    irreg_gdf = irreg_gdf.sort_values(["id", "timestamp_startTime_ts_cot"]).reset_index(drop=True)
    irreg_gdf["next_start"] = irreg_gdf.groupby("id")["timestamp_startTime_ts_cot"].shift(-1)
    mask = irreg_gdf["next_start"].notna()
    irreg_gdf.loc[mask, "timestamp_endTime_ts_cot"] = irreg_gdf.loc[mask, "next_start"]
    irreg_gdf = irreg_gdf.drop(columns="next_start")
    irreg_gdf["orig_duration"] = irreg_gdf["timestamp_endTime_ts_cot"] - irreg_gdf["timestamp_startTime_ts_cot"]
    irreg_gdf["split_slices"] = irreg_gdf.apply(
        lambda r: quarter_slices(r["timestamp_startTime_ts_cot"], r["timestamp_endTime_ts_cot"]),
        axis=1
    )
    irreg_gdf_exploded = irreg_gdf.explode("split_slices").reset_index(drop=True)
    irreg_gdf_exploded[["new_start", "new_end"]] = pd.DataFrame(
        irreg_gdf_exploded["split_slices"].tolist(), index=irreg_gdf_exploded.index
    )
    irreg_gdf_exploded["piece_duration"] = irreg_gdf_exploded["new_end"] - irreg_gdf_exploded["new_start"]
    irreg_gdf_exploded["pct_of_interval"] = (
        irreg_gdf_exploded["piece_duration"] / irreg_gdf_exploded["orig_duration"]
    )
    irreg_gdf_exploded['piece_duration_mins'] = (irreg_gdf_exploded["piece_duration"]
        .dt.total_seconds()
        .div(60))
    irreg_gdf = irreg_gdf_exploded.drop(
        columns=[
            "timestamp_startTime_ts_cot",
            "timestamp_endTime_ts_cot",
            "split_slices",
            "orig_duration",
            "piece_duration"
        ]
    )
    irreg_gdf = irreg_gdf.rename(
        columns={"new_start": "timestamp_startTime_ts_cot", "new_end": "timestamp_endTime_ts_cot"}
    )
    irreg_gdf = irreg_gdf.reset_index(drop=True)
    if irreg_gdf.crs != mvi_buffered_buffer.crs:
        irreg_gdf = irreg_gdf.to_crs(mvi_buffered_buffer.crs)

    irregs_intersections = gpd.overlay(irreg_gdf, mvi_buffered_buffer[['id_mvi', 'mvi_length', 'geometry']], how='intersection')
    irregs_intersections = irregs_intersections.to_crs("EPSG:3116").copy()
    irregs_intersections['intersection_length'] = irregs_intersections.geometry.length
    irregs_intersections['overlap_ratio'] = irregs_intersections['intersection_length'] / irregs_intersections['mvi_length']
    irregs_intersections = irregs_intersections[irregs_intersections['overlap_ratio'] > 0.2]
    irregs_intersections = irregs_intersections.to_crs("EPSG:4326").copy()
    irregs_intersections["timeframe"] = irregs_intersections["timestamp_startTime_ts_cot"].dt.floor("15min")

    non_agg_variables = ['type', 'timestamp_updateDateMillis_ts_cot', 'id', 'timeframe', 'timestamp_detectionDateMillis', 'geometry', 'overlap_ratio', 'intersection_length', 'irreg_id', 'pct_of_interval', 'timestamp_updateDateMillis', 'timestamp_startTime_ts_cot', 'alerts', 'timestamp_endTime_ts_cot', 'mvi_length', 'id_mvi', 'timestamp_detectionDateMillis_ts_cot']
    non_transformable_variables = ['trend','speed','jamLevel','severity','regularSpeed']
    countable_variables = ['causeType']
    transformable_variables = sorted(list(set(irregs_intersections.columns)-set(non_transformable_variables)-set(countable_variables)-set(non_agg_variables)))
    agg_dict = {}

    for col in transformable_variables:
        if col in irregs_intersections.columns:
            irregs_intersections[col] = irregs_intersections[col]*irregs_intersections["pct_of_interval"].clip(upper=1)*irregs_intersections["overlap_ratio"].clip(upper=1)
            agg_dict[col] = 'sum'

    for col in non_transformable_variables:
        if col in irregs_intersections.columns:
            irregs_intersections[f'{col}_weighted'] = irregs_intersections[col]*irregs_intersections["piece_duration_mins"]
            agg_dict[f'{col}_weighted'] = 'sum'
    agg_dict['piece_duration_mins'] = 'sum'

    dummies = pd.get_dummies(
        irregs_intersections[countable_variables].astype(str),
        prefix=countable_variables
    ).astype('int')
    for col in dummies.columns:
        agg_dict[col] = 'sum'
    irregs_intersections = pd.concat([irregs_intersections,dummies],axis=1)

    irregs_intersections = irregs_intersections.sort_values(['id_mvi','timeframe']).reset_index(drop=True)

    agg_irregs_intersections = irregs_intersections.groupby(['id_mvi','timeframe'], sort=False).agg(agg_dict)
    agg_irregs_intersections = agg_irregs_intersections.reset_index()


    for col in non_transformable_variables:
        agg_irregs_intersections[col] = agg_irregs_intersections[f'{col}_weighted']/agg_irregs_intersections['piece_duration_mins']
        agg_irregs_intersections = agg_irregs_intersections.drop([f'{col}_weighted'],axis=1)
    agg_irregs_intersections = agg_irregs_intersections.drop(['piece_duration_mins'],axis=1)
    agg_irregs_intersections.to_parquet(output_file_name, index=False)

In [ ]:
irreg_gdf = gpd.GeoDataFrame(pd.concat(irreg_gdf, ignore_index=True).fillna(0), crs=4326)
irreg_gdf["irreg_id"] = irreg_gdf.index
irreg_gdf = irreg_gdf.sort_values(["id", "timestamp_startTime_ts_cot"]).reset_index(drop=True)
irreg_gdf["next_start"] = irreg_gdf.groupby("id")["timestamp_startTime_ts_cot"].shift(-1)
mask = irreg_gdf["next_start"].notna()
irreg_gdf.loc[mask, "timestamp_endTime_ts_cot"] = irreg_gdf.loc[mask, "next_start"]
irreg_gdf = irreg_gdf.drop(columns="next_start")
irreg_gdf["orig_duration"] = irreg_gdf["timestamp_endTime_ts_cot"] - irreg_gdf["timestamp_startTime_ts_cot"]
irreg_gdf["split_slices"] = irreg_gdf.apply(
    lambda r: quarter_slices(r["timestamp_startTime_ts_cot"], r["timestamp_endTime_ts_cot"]),
    axis=1
)
irreg_gdf_exploded = irreg_gdf.explode("split_slices").reset_index(drop=True)
irreg_gdf_exploded[["new_start", "new_end"]] = pd.DataFrame(
    irreg_gdf_exploded["split_slices"].tolist(), index=irreg_gdf_exploded.index
)
irreg_gdf_exploded["piece_duration"] = irreg_gdf_exploded["new_end"] - irreg_gdf_exploded["new_start"]
irreg_gdf_exploded["pct_of_interval"] = (
    irreg_gdf_exploded["piece_duration"] / irreg_gdf_exploded["orig_duration"]
)
irreg_gdf_exploded['piece_duration_mins'] = (irreg_gdf_exploded["piece_duration"]
      .dt.total_seconds()
      .div(60))
irreg_gdf = irreg_gdf_exploded.drop(
    columns=[
        "timestamp_startTime_ts_cot",
        "timestamp_endTime_ts_cot",
        "split_slices",
        "orig_duration",
        "piece_duration"
    ]
)
irreg_gdf = irreg_gdf.rename(
    columns={"new_start": "timestamp_startTime_ts_cot", "new_end": "timestamp_endTime_ts_cot"}
)
irreg_gdf = irreg_gdf.reset_index(drop=True)


In [ ]:
if irreg_gdf.crs != mvi_buffered_buffer.crs:
    irreg_gdf = irreg_gdf.to_crs(mvi_buffered_buffer.crs)

irregs_intersections = gpd.overlay(irreg_gdf, mvi_buffered_buffer[['id_mvi', 'mvi_length', 'geometry']], how='intersection')
irregs_intersections = irregs_intersections.to_crs("EPSG:3116").copy()
irregs_intersections['intersection_length'] = irregs_intersections.geometry.length
irregs_intersections['overlap_ratio'] = irregs_intersections['intersection_length'] / irregs_intersections['mvi_length']
irregs_intersections = irregs_intersections[irregs_intersections['overlap_ratio'] > 0.2]
irregs_intersections = irregs_intersections.to_crs("EPSG:4326").copy()
irregs_intersections["timeframe"] = irregs_intersections["timestamp_startTime_ts_cot"].dt.floor("15min")

In [ ]:
non_agg_variables = ['type',
 'timestamp_updateDateMillis_ts_cot',
 'id',
 'timeframe',
 'timestamp_detectionDateMillis',
 'geometry',
 'overlap_ratio',
 'intersection_length',
 'irreg_id',
 'pct_of_interval',
 'timestamp_updateDateMillis',
 'timestamp_startTime_ts_cot',
 'alerts',
 'timestamp_endTime_ts_cot',
 'mvi_length',
 'id_mvi',
 'timestamp_detectionDateMillis_ts_cot']
non_transformable_variables = ['trend','speed','jamLevel','severity','regularSpeed']
countable_variables = ['causeType']
transformable_variables = sorted(list(set(irregs_intersections.columns)-set(non_transformable_variables)-set(countable_variables)-set(non_agg_variables)))
agg_dict = {}# {col:wavg for col in non_transformable_variables}

In [ ]:
for col in transformable_variables:
    if col in irregs_intersections.columns:
        irregs_intersections[col] = irregs_intersections[col]*irregs_intersections["pct_of_interval"].clip(upper=1)*irregs_intersections["overlap_ratio"].clip(upper=1)
        agg_dict[col] = 'sum'


In [ ]:
for col in non_transformable_variables:
    if col in irregs_intersections.columns:
        irregs_intersections[f'{col}_weighted'] = irregs_intersections[col]*irregs_intersections["piece_duration_mins"]
        agg_dict[f'{col}_weighted'] = 'sum'
agg_dict['piece_duration_mins'] = 'sum'

In [ ]:
dummies = pd.get_dummies(
    irregs_intersections[countable_variables].astype(str),
    prefix=countable_variables
).astype('int')
for col in dummies.columns:
    agg_dict[col] = 'sum'
irregs_intersections = pd.concat([irregs_intersections,dummies],axis=1)

In [ ]:
irregs_intersections = irregs_intersections.sort_values(['id_mvi','timeframe']).reset_index(drop=True)

In [ ]:
agg_irregs_intersections = irregs_intersections.groupby(['id_mvi','timeframe'], sort=False).agg(agg_dict)
agg_irregs_intersections = agg_irregs_intersections.reset_index(drop=True)

In [ ]:
for col in non_transformable_variables:
    agg_irregs_intersections[col] = agg_irregs_intersections[f'{col}_weighted']/agg_irregs_intersections['piece_duration_mins']
    agg_irregs_intersections = agg_irregs_intersections.drop([f'{col}_weighted'],axis=1)
agg_irregs_intersections = agg_irregs_intersections.drop(['piece_duration_mins'],axis=1)

In [ ]:
agg_irregs_intersections.reset_index()

In [ ]:
irregs_intersections = pd.concat([irregs_intersections,pd.get_dummies(
    irregs_intersections[countable_variables].astype(str),
    prefix=countable_variables
).astype('int')],axis=1)

In [ ]:
dummies = pd.get_dummies(
    irregs_intersections[countable_variables].astype(str),
    prefix=countable_variables
).astype('int')
for col in dummies.columns:
    agg_dict[col] = 'sum'
irregs_intersections = pd.concat([irregs_intersections,dummies],axis=1)

In [ ]:
df = irregs_intersections.copy()
dummies = pd.get_dummies(
    df[countable_variables].astype(str),
    prefix=countable_variables
)
df2 = pd.concat(
    [df[['id_mvi','timeframe']], dummies],
    axis=1
)
df2 = (
    df2
     .groupby(['id_mvi','timeframe'], sort=False)
     .sum()
     .reset_index()
)

In [ ]:
dummies

In [ ]:
agg_irregs_intersections.reset_index()